In [ ]:
"""
S01 MPO training — integrate notebooks 01–07 mission context.

Scenario:
  - Same mission profile as notebook 07 baseline overflight (50-target meridian grid,
    seeded clouds over the corridor, dual camera, attitude safety on, image quality)
  - Controller features selected via ControllerFeatureConfig (see cell below)
  - 10× baseline overflight warmup (notebook 07 policy, 2-D buffer) → train → eval
  - Preflight: inline feature checks + full ML training pytest suite

Verification: s01_utils/training_workflow.py
Artifacts: autonomous_control/models/nb-s01-08-<timestamp>/
Export: eval_best.mp4 in run directory
"""

In [ ]:
def setup_notebook_paths():
    """
    Configure Python paths and working directory for running the S01 training notebook.

    - Walks up directories from the current working directory until it finds the 'simulation' folder,
      which marks the backend root.
    - Changes the working directory to the backend root to ensure relative paths are correct.
    - Adds both the backend root and the S01 notebook utilities directory to sys.path for imports.

    This setup is required for importing backend modules and utility code in other cells.
    """
    import os
    import sys
    from pathlib import Path

    notebook_dir = Path.cwd()
    backend_root = notebook_dir
    for _ in range(6):
        if (backend_root / "simulation").is_dir():
            break
        backend_root = backend_root.parent
    os.chdir(backend_root)
    sys.path.insert(0, str(backend_root))
    _s01_dir = backend_root / "notebooks" / "s01"
    sys.path.insert(0, str(_s01_dir))
    print(f"backend_root={backend_root}")

setup_notebook_paths()

In [ ]:
import importlib

import s01_utils.training_workflow as tw

importlib.reload(tw)

if tw.run_s01_training_preflight_gate():
    print("Training preflight gate passed (inline checks + pytest suite).")

In [ ]:
from autonomous_control.feature_selection import ControllerFeatureConfig

# ── EDIT HERE: controller observation features ───────────────────────────────
# Keys must exist on SimulationTimestepState (see autonomous_control/feature_selection.py).
FEATURE_CONFIG = ControllerFeatureConfig(
    attitude_keys=(
        "body_z_angle_rad",
        "omega_sat_rad_s",
    ),
    orbit_keys=(
        "theta_orbit_rad",
    ),
    vision_keys=(
        "camera_observation_line_codes",
        "secondary_camera_observation_line_codes",
    ),
)

WORKFLOW_CONFIG = tw.TrainingWorkflowConfig(
    seed=7,
    train_episodes=100,
    feature_config=FEATURE_CONFIG,
)

# Resolve secondary camera bin count for layout tables (same mission profile as training).
_secondary_bins = int(
    tw.build_s01_training_mission_setup(seed=WORKFLOW_CONFIG.seed)
    .resolve(require_camera=True)
    .secondary_camera_observation_line_n_bins
)
tw.display_feature_tables(FEATURE_CONFIG, secondary_camera_bins=_secondary_bins)

In [ ]:
setup = tw.build_training_workflow_setup(WORKFLOW_CONFIG)
tw.print_training_setup_summary(setup)
tw.display_feature_snapshot_tables(setup)

In [ ]:
result = tw.run_training_workflow(setup, show_progress=True)

In [ ]:
tw.print_training_kpis(result)

In [ ]:
from utils.notebook.video import init_video_cell, play_saved_video

init_video_cell()

from utils.notebook.video import init_video_cell, play_saved_video

init_video_cell()
tw.display_training_artifacts(result)
video_path = result.artifact_paths["eval_best_video"]
if video_path.exists():
    play_saved_video(video_path)